# Simulate RI Logit

## Packages

In [1]:
import numpy as np

from scipy.optimize import minimize
from src.simulation.simulate_logit import SimulateRILogit

## Global Parameters

#TODO find a simulation that gives rise to marg all interiors and then estimate the logit MLE. 

In [3]:
J = 3 # Number of products
N = 3 # Number of states per products
llambda = 0.1 # Information Cost
I = 1 # Number of Individuals
n_sim = 100000 # Number of simulation for each individuals
state = (0, 1, 0)

## Simulate Individuals Characteristics

## Simulate Products Characteristics

## Obtain Utility from Characteristics

To assess the performance of our estimators we have to get payoff and prior matrix as function of product characteristics $X_j$ individual characteristics $X_i$ and preferences $\theta \in \Theta$ to be estimated.

$u_{ij} = \theta_1 X_i + \theta_2 X_j $ (additively separable) 

$u_{ij} = h(\theta, X_i, X_j) $

If we want to model heterogeneity, we would compute utilities from different $\theta$.

In [ ]:
def utility(x_i: np.array, x_j: np.array, theta: np.array, form: str):
    """Encode the utility function of a homogenous set of
    individuals with preferences theta

    Args:
        x_i (np.array): individuals characteristics
        x_j (np.array): products characteristics
        theta (np.array): preferences
        form (str): if additive, multiplicative, etc
    """
    if form == "additive":
        return theta[0, :] @ x_i + theta[1, :] @ x_j

## Simulate Logit Choice

Using Blahut–Arimoto solver

In [6]:
list_classes_BA = [
    SimulateRILogit(u_mat=u_mat, ppi=ppi, llambda=llambda)
    for u_mat, ppi in zip(list_u_mat, list_ppi)
]
list_indexes_BA = [cl.get_states().index(state) for cl in list_classes_BA]
list_choices_BA = [cl.simulate(n_sim=n_sim, states=state) for cl in list_classes_BA]
list_dist_BA = [cl.get_logit()[index] for cl, index in zip(list_classes_BA, list_indexes_BA)]
list_marg_BA = [cl.get_marg() for cl in list_classes_BA]

Blahut–Arimoto Solver:   0%|          | 25/10000 [00:00<00:03, 2877.23iter/s, Error=[4.69684624e-13]]


In [7]:
list_marg_BA

[array([[0.06652155],
        [0.57955377],
        [0.35392469]])]

In [48]:
list_classes_SQP = [
    SimulateRILogit(u_mat=u_mat, ppi=ppi, llambda=llambda, method="SQP")
    for u_mat, ppi in zip(list_u_mat, list_ppi)
]
list_indexes_SQP = [cl.get_states().index(state) for cl in list_classes_SQP]
list_choices_SQP = [cl.simulate(n_sim=n_sim, states=state) for cl in list_classes_SQP]
list_dist_SQP = [cl.get_logit()[index] for cl, index in zip(list_classes_SQP, list_indexes_SQP)]

SQP Solver:   0%|          | 1/10000 [00:00<01:03, 156.52iter/s, Error=0.00237]


ValueError: Invalid dimensions (0,).

## Estimation

In [ ]:
n_j = list_choices_BA[0][1] * n_sim 

u_init = np.ones(J)
q_init = np.full(J, 1/J)

def constraint_q(params):
    q = params[J:]
    return np.sum(q) - 1

def log_likelihood(params, observations):
    theta = params[:J]  
    q = params[J:]

    b = np.exp( / llambda)

    denominator = np.sum(b * q)
    log_likelihood_value = np.sum(n_j * (np.log(b) + np.log(q) - np.log(denominator)))

    return -log_likelihood_value 

params_init = np.concatenate([u_init, q_init])

bounds = [(1e-6, None)] * J + [(1e-6, 1-1e-6)] * J

constraints = [{"type": "eq", "fun": constraint_q}]

result = minimize(
    log_likelihood, params_init, bounds=bounds, constraints=constraints, options={'disp':True}, method="SLSQP"
)

u_est = result.x[:J]
q_est = result.x[J:]

print("Estimated u:", u_est)
print("Estimated q:", q_est)

Optimization terminated successfully    (Exit mode 0)
            Current function value: 27290.1142056978
            Iterations: 26
            Function evaluations: 229
            Gradient evaluations: 26
Estimated u: [1.32547035 1.96120807 1.65221369]
Estimated q: [0.56118174 0.17297885 0.26583941]


/Users/maximecoulet/miniconda3/envs/RIProject/lib/python3.12/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/var/folders/7l/_s01npq1225c41_90vkwthlc0000gn/T/ipykernel_3956/630352060.py:14: RuntimeWarning: overflow encountered in exp
  b = np.exp(u / llambda)
/var/folders/7l/_s01npq1225c41_90vkwthlc0000gn/T/ipykernel_3956/630352060.py:17: RuntimeWarning: invalid value encountered in subtract
  log_likelihood_value = np.sum(n_j * (np.log(b) + np.log(q) - np.log(denominator)))


In [52]:
result.fun

np.float64(46345.04352038879)